# Intent & Advertiser Routing Classifier

> Routes French user search queries to advertiser verticals using two approaches: a low-latency TF-IDF baseline and an LLM-powered semantic classifier with structured output.

In [ ]:
import pandas as pd
import numpy as np
import time
import os

from sentence_transformers import SentenceTransformer

from typing import List, Literal
from pydantic import BaseModel, Field
from openai import OpenAI

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [2]:
queries = pd.read_csv("data/queries.csv")

## Data Analysis

In [3]:
tokens = queries["query"].str.lower().str.findall(r"\w+")
queries["n_tokens"] = tokens.map(len)
queries["n_chars"] = queries["query"].map(len)

In [4]:
queries.head()

,query,category,n_tokens,n_chars
0,acheter chaussures en ligne,E-commerce,4,27
1,promo nike homme,E-commerce,3,16
2,soldes vêtements femme,E-commerce,3,22
3,code promo amazon,E-commerce,3,17
4,acheter canapé pas cher,E-commerce,4,23


We have a supervised classification dataset. 

- Input: query (user search query, text)
- Output: category (advertiser category)

In [5]:
queries.describe(include="all")

,query,category,n_tokens,n_chars
count,147,147,147.000000,147.000000
unique,68,10,NaN,NaN
top,acheter chaussures en ligne,E-commerce,NaN,NaN
freq,6,91,NaN,NaN
mean,NaN,NaN,3.190476,21.346939
std,NaN,NaN,0.644612,3.968460
min,NaN,NaN,2.000000,12.000000
25%,NaN,NaN,3.000000,18.000000
50%,NaN,NaN,3.000000,22.000000
75%,NaN,NaN,4.000000,25.000000


In [6]:
queries["category"].value_counts()

category
E-commerce       91
Travel            9
Insurance         7
Finance           7
Telecom           7
Health            7
Electronics       5
Home Services     5
Automotive        5
Education         4
Name: count, dtype: int64

We notice that 

1)  There is an important class imbalance. The ecommerce category dominates the dataset. 

    The model may learn to favor ecommerce heavily because predicting it minimizes overall error. It may also completely ignore some categories. Accuracy alone is not a sufficient evaluation metric. 

2) more than 50% of rows are duplicates; data leakage risks > if the same query appears in both train and test sets, the model can memorize it. To mitigate this issue, train-test splits must be performed at the unique query level.

3) Queries are short (21.34 characters on average). ANOVA test: p-value = 0.21. There is no statistically significant difference in query length between categories. Character/word length is not very informative and won’t add meaningful signal for category prediction > we drop it. 

4) The dataset is extremely small (147 rows, 68 unique queries) > limits the model ability to generalize and increases the risk of overfitting. Performance measured on this dataset may overestimate real-world performance.

## Baseline Model : TF-IDF + Logistic Regression

This is a classic strong baseline for short text classification. It is fast at inference (usually a few milliseconds per query on CPU), and works well even with small datasets because it’s linear and regularized.

We create a scikit-learn Pipeline object that chains multiple steps so data flows sequentially: raw text → vectorization → classifier.

In [12]:
queries = queries.drop(columns=["n_tokens","n_chars"])

In [13]:
# we avoid leakage from duplicated queries by splitting at unique-query level
df_unique = queries.drop_duplicates(subset=["query"]).copy()

In [14]:
# we stratify
train_df, test_df = train_test_split(df_unique, test_size=0.2, random_state=42, stratify=df_unique["category"])

Stratifying ensures that the train and test sets preserve the same category distribution as the original dataset, which is especially important here due to severe class imbalance. Without stratification, some minority categories might appear only in the train or test set, leading to unreliable evaluation and biased performance estimates.

In [15]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [16]:
X_train = train_df["query"].values
y_train = train_df["category"].values
X_test  = test_df["query"].values
y_test  = test_df["category"].values

1. **TfidfVectorizer** converts text into numerical features using TF-IDF weighting

TF = how often a word appears in a document
IDF = how rare that word is across documents
product > gives higher weight to most informative words

2. **Logistic Regression** classifier

eLogistic Regression learns a linear decision boundary in the high-dimensional feature space returned by TF-IDF. It estimates the probability of a sample belonging to a given class using the logistic (sigmoid) function, predicting the label with the highest probability.

We use class_weight=”balanced” to automatically reweight classes inversely proportionally to their frequency.

In [17]:
model_word = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=1
    )),
    ("clf", LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ))
])

model_word.fit(X_train, y_train)
pred = model_word.predict(X_test)

print(classification_report(y_test, pred, digits=3))

               precision    recall  f1-score   support

   Automotive      0.000     0.000     0.000         1
   E-commerce      0.600     1.000     0.750         3
    Education      0.000     0.000     0.000         1
  Electronics      0.000     0.000     0.000         1
      Finance      1.000     1.000     1.000         2
       Health      0.000     0.000     0.000         1
Home Services      1.000     1.000     1.000         1
    Insurance      0.667     1.000     0.800         2
      Telecom      1.000     1.000     1.000         1
       Travel      0.000     0.000     0.000         1

     accuracy                          0.643        14
    macro avg      0.427     0.500     0.455        14
 weighted avg      0.510     0.643     0.561        14



c:\Users\odend\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\odend\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\odend\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

## Evaluation

We have strong class imbalance and small dataset size > accuracy alone is not sufficient (it can be dominated by ecommerce)

F1 score : valuates performance equally across all advertiser categories

top 3 accuracy : relevant in an advertising context where the system can surface multiple candidate categories; downstream ranking can select the best advertiser

In [18]:
from sklearn.metrics import accuracy_score, f1_score

In [19]:
# top-1 accuracy
acc = accuracy_score(y_test, pred)

# macro F1 (treats all categories equally)
macro_f1 = f1_score(y_test, pred, average="macro")

# Top-3 accuracy
proba = model_word.predict_proba(X_test)
labels = model_word.named_steps["clf"].classes_

def top_k_accuracy(y_true, proba, labels, k=3):
    correct = 0
    for i in range(len(y_true)):
        topk_idx = np.argsort(-proba[i])[:k]
        topk_labels = labels[topk_idx]
        if y_true[i] in topk_labels:
            correct += 1
    return correct / len(y_true)

top3 = top_k_accuracy(y_test, proba, labels, k=3)

In [20]:
print(f"Accuracy: {acc:.3f}")
print(f"Macro F1: {macro_f1:.3f}")
print(f"Top-3 Accuracy: {top3:.3f}")

Accuracy: 0.643
Macro F1: 0.455
Top-3 Accuracy: 0.786


We focus on 3 simple meaningful metrics:
- **accuracy** (simple, intuitive but inflated by class imbalance)
- **macro F1 score** (corrects for class imbalance, more representative of real perf)
- **top 3 accuracy** (relevant for ad systems);
 
baseline perf: Accuracy is decent (0.643; top3 = 0.786) but macro F1 tells the full story : performance across categories is uneven.

The model performs on categories with strong lexical markers (finance, insurance, telecom) but fails on automotive, health, education

> dataset is too small for stable per-class evaluation
> TF-IDF + LogReg struggles on categories with weak or ambiguous keywords

**production risks** : poor generalization to new queries, vocabulary drift, category expansion problem

Our top3 accuracy is significantly higher than top1. The model is better at candidate generation than strict classification. 
It is acceptable in an advertising system if:
- ranking layer can choose best advertiser
- pricing and auction optimize final outcome

But minority categories remain at risk of under-exposure.

**improving performance ?**

we thought about + tested pretrained embeddings (sentence-transformers). larger corpora could have led to better generalization. However, increase in perf was not significative (same metrics as TF-IDF) > probably due to short queries (explicit keywords; low semantic ambiguity)

we also considered data augmentation techniques (synonym replacement, paraphrasing) to increase training diversity + active learning to max improvement per labeled example.

## sentence-transformers embeddings test

In [43]:
# load small, fast embedding model
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# encode once (heavier part)
E_train = embedder.encode(X_train, normalize_embeddings=True)
E_test  = embedder.encode(X_test,  normalize_embeddings=True)

E_train.shape, E_test.shape

((54, 384), (14, 384))

In [44]:
emb_clf = LogisticRegression(
    max_iter=2000,
    class_weight="balanced"
)

emb_clf.fit(E_train, y_train)

pred_emb = emb_clf.predict(E_test)
proba_emb = emb_clf.predict_proba(E_test)
labels_emb = emb_clf.classes_

In [45]:
acc = accuracy_score(y_test, pred_emb)
macro_f1 = f1_score(y_test, pred_emb, average="macro")
top3 = top_k_accuracy(y_test, proba_emb, labels_emb, k=3)

print(f"Accuracy (emb+logreg): {acc:.3f}")
print(f"Macro F1  (emb+logreg): {macro_f1:.3f}")
print(f"Top-3 Acc (emb+logreg): {top3:.3f}")

Accuracy (emb+logreg): 0.643
Macro F1  (emb+logreg): 0.442
Top-3 Acc (emb+logreg): 0.786


# LLM-powered advertiser classification

Our most promising approach came with the implementation of LLM-powered classification.

We perform advertiser category prediction through a single inference call. At startup, the openai client is initialized once. For each incoming query, the system sends a prompt containing the closed set of advertiser categories and uses Pydantic structured output to enforce a strict response schema (category, top-3 categories, confidence). The model (gpt-4o-mini) returns a validated, parsed object, and end-to-end latency is measured per query using a simple timing wrapper.

In [ ]:
api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=api_key)

categories = [
    "E-commerce", "Travel", "Insurance", "Finance", "Telecom",
    "Health", "Electronics", "Home Services", "Automotive", "Education"
]

CategoryLiteral = Literal[tuple(categories)]

intents = ["transaction", "deal_seeking", "research", "subscription_service", "local_service"]

# structured output schema
class CategoryIntentPrediction(BaseModel):
    
    category: CategoryLiteral
    top3: List[CategoryLiteral]
    intent: Literal[tuple(intents)]
    confidence: float = Field(..., ge=0.0, le=1.0)

In [48]:
system = f"""
You are an advertiser routing classifier.

Choose:
- category from: {categories}
- intent from: {intents}

Intent definitions:
transaction=buy/book now; deal_seeking=discount/promo; research=compare/reviews;
subscription_service=plans/contracts (forfait/assurance/prêt); local_service=repair/help nearby.

Return JSON only following the schema.
"""

def predict_llm(query: str):

    t0 = time.perf_counter()

    response = client.responses.parse(
        model="gpt-4o-mini",
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": query},
        ],
        text_format=CategoryIntentPrediction,
    )

    result = response.output_parsed

    latency_ms = (time.perf_counter() - t0) * 1000

    return {
        "query": query,
        "pred": result.category,
        "top3": result.top3,
        "intent": result.intent,
        "confidence": result.confidence,
        "latency_ms": latency_ms,
    }

In [49]:
for q in queries["query"].sample(5):
    print(predict_llm(q))

{'query': 'acheter vélo en ligne', 'pred': 'E-commerce', 'top3': ['E-commerce', 'Automotive', 'Health'], 'intent': 'transaction', 'confidence': 0.95, 'latency_ms': 1711.6036000079475}
{'query': 'mutuelle optique', 'pred': 'Health', 'top3': ['Insurance', 'Health', 'Finance'], 'intent': 'subscription_service', 'confidence': 0.85, 'latency_ms': 1389.010499988217}
{'query': 'acheter canapé pas cher', 'pred': 'E-commerce', 'top3': ['E-commerce', 'Home Services', 'Electronics'], 'intent': 'deal_seeking', 'confidence': 0.9, 'latency_ms': 1601.3009999878705}
{'query': 'vacances pas cher', 'pred': 'Travel', 'top3': ['Travel', 'E-commerce', 'Finance'], 'intent': 'deal_seeking', 'confidence': 0.85, 'latency_ms': 1278.8976999581791}
{'query': 'acheter cadeau anniversaire', 'pred': 'E-commerce', 'top3': ['E-commerce', 'Health', 'Electronics'], 'intent': 'transaction', 'confidence': 0.95, 'latency_ms': 1447.143799974583}


Time constraints didnt allow for strict metrics calculation but empirically we can affirm that top3 accuracy on our dataset is close to 100%.
The approach is strong when labels are scarce and queries are semantically varied; 
> good generalization to unseen phrasing
> quick to iterate on (prompt tweaks, taxonomy changes) without retraining.
 
In an advertiser routing system, we believe the model role is typically candidate generation, not final ranking > map a user query to one or several advertiser verticals so downstream systems (auction, ranking, budget constraints) can decide which specific advertiser to show.

**downsides:**

- dependence on an external API (availability, privacy constraints, vendor lock-in)
- higher and more variable latency (typically 1-2s with occasional spikes); acceptable depending on the architecture (maybe not for real-time ad serving in search or display ads)
- ongoing per-query cost; quick api cost estimate : 200 input tokens ($0.15/M) + 50 output tokens ($0.6/M) ~ $0.00006/query

TF-IDF + LogReg is the opposite: essentially free per query, ultra-low latency and stable, easy to deploy anywhere, but it’s brittle to paraphrases, needs labeled data to improve, and tends to over-rely on keywords (worse generalization).

**potential improvements:**

- hybrid routing (combine TF-IDF to generate top-k + LLM to rerank; lower cost + latency as we scale)
- keep the instructions/categories as cached/persistent context (cached input is 2x cheaper)
- confidence calibration/fallback (poor variance for now)


## Intent modeling

We now explore user intent classification. We start with the following taxonomy:
intents = ["transaction", "deal_seeking", "research", "subscription_service", "local_service"]

Empirically, we observe relations emerging (linked to extremely small dataset so not significant):

- transaction > mostly ecommerce/automotive
- deal seeking > insurance/electronics
- research > finance/education

However, the pattern is:

- category determines what vertical
- intent determines where in the funnel the user is

Intent improves advertiser matching:

- funnel-aware advertiser selection
- budget optimization
- better ranking features; less ambiguity
- potential for different model routing : high-confidence transaction intent > fast model ; ambiguous research intent > LLM

Advertiser and Intent classifiers encode orthogonal signals > system goes from simple classifier to query understanding model